# Modelo genérico · Deep Q-Network (DQN) com replay de experiências e rede-alvo

**Template reutilizável** de aprendizagem por reforço para qualquer ambiente **Gymnasium** com observação vetorial e **ações discretas**: rede Q, rede-alvo com atualização suave, replay de experiências, política ε-gulosa e avaliação gulosa separada do treino.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 17. Aprendizagem por reforço › cartão *Deep Q-Network (DQN)* (`/modelar/aprendizagem-por-reforco/#m-dqn`) |
| **Origem** | `C3_W3_A1_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; pouso do *Lunar Lander*), revisado, corrigido e generalizado |
| **Tipo de problema** | Aprendizagem por reforço · baseada em valor, *off-policy* |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `gymnasium`, `tensorflow` 2.16+ com `keras` 3 |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. O padrão é `CartPole-v1`, que não exige dependências extras. Para o ambiente do original, instale `pip install swig "gymnasium[box2d]"` e use `ENV_ID = "LunarLander-v3"` (a versão `v2` do `gym` foi descontinuada).
3. O treino é a parte demorada: reduza `NUM_EPISODIOS` para testar o fluxo e aumente para treinar de verdade.

### O que mudou em relação ao notebook original
- **API do Gymnasium.** O original usava `gym` 0.24, descontinuado: `reset()` devolvia só o estado, `step()` devolvia 4 valores e `render(mode=...)` não existe mais. Agora: `reset(seed=...)` → `(estado, info)` e `step()` → `(estado, recompensa, terminated, truncated, info)`.
- **Erro de alvo corrigido:** o `done` do `gym` antigo misturava fim real (`terminated`) com corte por limite de tempo (`truncated`). Zerar o termo `γ·max Q` num corte de tempo ensina à rede que o estado não tem futuro. Agora o buffer guarda `terminated` e o episódio acaba em `terminated or truncated`.
- O treino salvo no original foi **interrompido no episódio 236** (média −113, longe dos 200 da meta), e o gráfico e o vídeo nunca foram gerados. O template guarda os pesos do melhor momento do treino (*checkpoint*) e inclui uma avaliação **gulosa** em episódios separados (média e desvio), como o cartão pede.
- Todas as funções do `utils.py` (ausente) foram escritas no notebook: `get_action`, `check_update_conditions`, `get_experiences`, `update_target_network`, `get_new_eps`, `plot_history`. A meta de pontos vem do próprio ambiente (`reward_threshold`), não fixa em 200.
- **Hiperparâmetros por ambiente:** os valores do original foram ajustados para o Lunar Lander; aplicados ao CartPole, o agente não aprende (ε decai por episódio rápido demais). O padrão passou a ser o do cartão (decaimento de ε por passo, aquecimento do buffer, atualização a cada passo, τ = 0,005), e os valores do original ficam documentados na configuração. Perda configurável: `"huber"` (cartão) ou `"mse"` (original). O modelo é salvo em `.keras` (o `.h5` é legado no Keras 3). `pyvirtualdisplay` e o vídeo em `.mp4` (`imageio`) saíram: o vídeo opcional é um GIF gerado com Pillow.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro ambiente.

ENV_ID = "CartPole-v1"       # qualquer ambiente Gymnasium com ações discretas; ex.: "LunarLander-v3", "Acrobot-v1"
SEMENTE = 0

# --- hiperparâmetros ---
# Padrão: valores que o cartão do sistema usa (e a auditoria testou) no CartPole.
# Valores do original, ajustados para o LunarLander: GAMMA = 0.995, ALPHA = 1e-3, NUM_STEPS_FOR_UPDATE = 4,
# TAU = 1e-3, MEMORY_SIZE = 100_000, CAMADAS = [64, 64], PERDA = "mse", DECAIMENTO_EPSILON = "episodio",
# E_DECAY = 0.995, E_MIN = 0.01, AQUECIMENTO = MINIBATCH_SIZE.
MEMORY_SIZE = 50_000         # tamanho do replay buffer
GAMMA = 0.99                 # fator de desconto
ALPHA = 5e-4                 # taxa de aprendizado
NUM_STEPS_FOR_UPDATE = 1     # faz uma atualização a cada C passos
MINIBATCH_SIZE = 64
AQUECIMENTO = 1_000          # transições no buffer antes da primeira atualização
TAU = 0.005                  # atualização suave da rede-alvo
CAMADAS = [128, 128]
PERDA = "huber"              # "huber" (cartão) ou "mse" (original)
DECAIMENTO_EPSILON = "passo" # "passo" (linear em PASSOS_DECAIMENTO, como no cartão) ou "episodio" (× E_DECAY, original)
E_INICIAL, E_MIN = 1.0, 0.05
PASSOS_DECAIMENTO = 10_000   # modo "passo": passos até ε chegar a E_MIN
E_DECAY = 0.995              # modo "episodio": ε ← max(E_MIN, E_DECAY · ε) ao fim de cada episódio

# --- treino e avaliação ---
NUM_EPISODIOS = 1000
MAX_PASSOS = 1000            # limite por episódio (o ambiente pode ter um limite menor)
JANELA_MEDIA = 100           # "resolvido" = média dos últimos JANELA_MEDIA episódios >= META_PONTOS
META_PONTOS = None           # None = reward_threshold do ambiente (CartPole: 475; LunarLander: 200)
JANELA_CHECKPOINT = 20       # guarda os pesos da melhor média móvel nesta janela e os restaura no fim
EPISODIOS_AVALIACAO = 10
GRAVAR_GIF = False           # True gera um GIF do agente treinado (exige pygame para renderizar)

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")   # o laço de treino usa tf.GradientTape

import random
import time
from collections import deque, namedtuple

import gymnasium as gym
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

random.seed(SEMENTE); np.random.seed(SEMENTE); tf.random.set_seed(SEMENTE)
print("Gymnasium", gym.__version__, "| TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 2. O ambiente
Todo ambiente Gymnasium segue o laço agente-ambiente: em cada passo *t*, o agente observa o estado $S_t$, escolhe uma ação $A_t$ pela política π, recebe a recompensa $R_t$ e passa ao estado $S_{t+1}$. Aqui se descobrem, do próprio ambiente, o tamanho do estado e o número de ações (no original, 8 e 4).

In [ ]:
env = gym.make(ENV_ID)
state_size = env.observation_space.shape
if not isinstance(env.action_space, gym.spaces.Discrete):
    raise TypeError("DQN exige ações discretas; para ações contínuas, veja PPO ou SAC no cartão de Policy Gradient.")
num_actions = int(env.action_space.n)       # o Gymnasium devolve np.int64; o Keras exige int
env.action_space.seed(SEMENTE)              # reset(seed=...) não semeia action_space.sample()
meta = META_PONTOS if META_PONTOS is not None else (env.spec.reward_threshold or float("inf"))
print("estado:", state_size, "| ações:", num_actions, "| meta de pontos:", meta,
      "| limite de passos do ambiente:", env.spec.max_episode_steps)

# Um passo de exemplo (equivale ao display_table do original)
current_state, _ = env.reset(seed=SEMENTE)
action = 0
next_state, reward, terminated, truncated, _ = env.step(action)
display(pd.DataFrame({"estado atual": current_state, "próximo estado": next_state}).T.round(4)
        .assign(ação=action, recompensa=reward, terminated=terminated, truncated=truncated))

## 3. Deep Q-Learning
Com estados e ações discretos, a equação de Bellman estima a função ação-valor iterativamente: $Q_{i+1}(s,a) = R + \gamma\max_{a'}Q_i(s',a')$. Com estados contínuos, uma rede neural aproxima $Q(s,a)\approx Q^*(s,a)$, treinada para reduzir o erro quadrático dessa equação.

Duas técnicas evitam a instabilidade:
- **Rede-alvo** $\hat Q$: gera os alvos $y = R + \gamma\max_{a'}\hat Q(s',a';w^-)$ e acompanha a rede principal devagar, por atualização suave $w^- \leftarrow \tau w + (1-\tau)w^-$, com $\tau \ll 1$.
- **Replay de experiências:** as tuplas $(S_t, A_t, R_t, S_{t+1})$ vão para um buffer, e o treino usa mini-lotes aleatórios dele, quebrando a correlação entre passos consecutivos.

In [ ]:
def criar_rede_q():
    return keras.Sequential([keras.Input(shape=state_size)]
                            + [layers.Dense(u, activation="relu") for u in CAMADAS]
                            + [layers.Dense(num_actions, activation="linear")])


q_network = criar_rede_q()
target_q_network = criar_rede_q()
optimizer = keras.optimizers.Adam(learning_rate=ALPHA)
q_network.summary()
assert q_network.output_shape == (None, num_actions)

experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

## 4. Perda
$$y_j = \begin{cases} R_j & \text{se o episódio termina em } j+1\\ R_j + \gamma\max_{a'}\hat Q(s_{j+1},a') & \text{caso contrário}\end{cases}$$

Em uma linha, com o indicador de término: `y = R + γ·max Q̂(s′)·(1 − done)`. Aqui `done` é **`terminated`**: um corte por limite de tempo (`truncated`) não zera o futuro.

In [ ]:
funcao_perda = keras.losses.Huber() if PERDA == "huber" else keras.losses.MeanSquaredError()


def compute_loss(experiences, gamma, q_network, target_q_network):
    """Perda entre os alvos y e Q(s, a) das ações tomadas, para um mini-lote."""
    states, actions, rewards, next_states, done_vals = experiences
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    y_targets = rewards + gamma * max_qsa * (1 - done_vals)
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(tf.shape(q_values)[0]), tf.cast(actions, tf.int32)], axis=1))
    return funcao_perda(y_targets, q_values)


# Conferência com um cálculo em NumPy num lote aleatório.
s = np.random.rand(5, *state_size).astype("float32"); s2 = np.random.rand(5, *state_size).astype("float32")
a = np.random.randint(0, num_actions, 5); r = np.random.rand(5).astype("float32"); d = np.array([0, 1, 0, 0, 1], "float32")
y_np = r + GAMMA * target_q_network(s2).numpy().max(1) * (1 - d)
q_np = q_network(s).numpy()[np.arange(5), a]
esperado = np.mean((y_np - q_np) ** 2) if PERDA == "mse" else float(keras.losses.Huber()(y_np, q_np))
assert np.isclose(float(compute_loss((s, a.astype("float32"), r, s2, d), GAMMA, q_network, target_q_network)), esperado, rtol=1e-4)
print("compute_loss conferida")

## 5. Funções auxiliares (antes no `utils.py` do curso)

In [ ]:
def get_action(q_values, epsilon=0.0):
    """ε-gulosa: com probabilidade ε, ação aleatória; senão, a de maior Q."""
    if random.random() > epsilon:
        return int(np.argmax(q_values.numpy()[0]))
    return random.randrange(num_actions)


def check_update_conditions(t, num_steps_upd, memory_buffer):
    """Atualiza a cada num_steps_upd passos (t = contador global de passos), depois do aquecimento do buffer."""
    return (t + 1) % num_steps_upd == 0 and len(memory_buffer) >= max(MINIBATCH_SIZE + 1, AQUECIMENTO)


def get_experiences(memory_buffer):
    lote = random.sample(memory_buffer, k=MINIBATCH_SIZE)
    para = lambda campo, dt="float32": tf.convert_to_tensor(np.array([getattr(e, campo) for e in lote]), dtype=dt)
    return para("state"), para("action"), para("reward"), para("next_state"), para("done")


def update_target_network(q_network, target_q_network):
    """Atualização suave: w⁻ ← τ·w + (1 − τ)·w⁻."""
    for w_alvo, w in zip(target_q_network.weights, q_network.weights):
        w_alvo.assign(TAU * w + (1.0 - TAU) * w_alvo)


def get_new_eps(epsilon, passos_totais=None):
    """Modo "episodio" (original): multiplica por E_DECAY. Modo "passo" (cartão): decai linearmente com os passos."""
    if DECAIMENTO_EPSILON == "passo" and passos_totais is not None:
        return max(E_MIN, E_INICIAL - (E_INICIAL - E_MIN) * passos_totais / PASSOS_DECAIMENTO)
    return max(E_MIN, E_DECAY * epsilon)


@tf.function
def q_valores(estados):
    """Q(s, ·) compilado: a escolha de ação acontece a cada passo e precisa ser rápida."""
    return q_network(estados, training=False)


@tf.function
def agent_learn(experiences, gamma):
    """Um passo de gradiente na rede Q e a atualização suave da rede-alvo."""
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)
    gradients = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))
    update_target_network(q_network, target_q_network)

## 6. Treino
O algoritmo do original, linha a linha: buffer com capacidade `MEMORY_SIZE`; rede-alvo começa igual à rede Q; em cada passo, ação ε-gulosa, transição guardada no buffer e, a cada `NUM_STEPS_FOR_UPDATE` passos (depois do aquecimento), um mini-lote aleatório para o gradiente; ε decai por passo ou por episódio (`DECAIMENTO_EPSILON`); para quando a média dos últimos `JANELA_MEDIA` episódios atinge a meta.

**Checkpoint do melhor modelo:** o DQN costuma regredir depois de um bom momento (esquecimento catastrófico). O laço guarda os pesos da melhor média móvel de `JANELA_CHECKPOINT` episódios e, se a meta não for atingida, restaura-os no fim. Na validação, uma semente chegou a média 315 no episódio 600 e caiu para 43 no 1500; sem o checkpoint, o modelo final seria o pior.

**Por que o decaimento por passo é o padrão:** episódios curtos no começo (no CartPole, ~20 passos) fazem o ε por episódio cair antes de o buffer ter experiência suficiente. Na validação deste template, os valores do original aplicados ao CartPole terminaram 700 episódios com média ~10 (política pior que a aleatória). Com os valores padrão e o checkpoint, três sementes (0, 1 e 2) chegaram ao retorno guloso máximo do ambiente, 500 ± 0 em 10 episódios de avaliação, em cerca de 1,5 minuto de CPU cada.

In [ ]:
start = time.time()
total_point_history = []
epsilon = E_INICIAL
memory_buffer = deque(maxlen=MEMORY_SIZE)
target_q_network.set_weights(q_network.get_weights())
resolvido, passos_totais = False, 0
melhor_media, melhores_pesos, melhor_episodio = -np.inf, q_network.get_weights(), 0

for i in range(NUM_EPISODIOS):
    state, _ = env.reset(seed=SEMENTE + i)
    total_points = 0.0
    for t in range(MAX_PASSOS):
        q_values = q_valores(np.expand_dims(state, axis=0).astype("float32"))
        action = get_action(q_values, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        memory_buffer.append(experience(state, action, reward, next_state, float(terminated)))
        if check_update_conditions(passos_totais, NUM_STEPS_FOR_UPDATE, memory_buffer):
            agent_learn(get_experiences(memory_buffer), GAMMA)
        state = next_state
        total_points += reward
        passos_totais += 1
        if DECAIMENTO_EPSILON == "passo":
            epsilon = get_new_eps(epsilon, passos_totais)
        if terminated or truncated:
            break
    total_point_history.append(total_points)
    media = np.mean(total_point_history[-JANELA_MEDIA:])
    media_ck = np.mean(total_point_history[-JANELA_CHECKPOINT:])
    if len(total_point_history) >= JANELA_CHECKPOINT and media_ck > melhor_media:
        melhor_media, melhores_pesos, melhor_episodio = media_ck, q_network.get_weights(), i + 1
    if DECAIMENTO_EPSILON == "episodio":
        epsilon = get_new_eps(epsilon)
    if (i + 1) % max(1, NUM_EPISODIOS // 20) == 0 or i == 0:
        print(f"Episódio {i + 1:5d} | média dos últimos {JANELA_MEDIA}: {media:8.2f} | ε = {epsilon:.3f}")
    if len(total_point_history) >= JANELA_MEDIA and media >= meta:
        print(f"\nAmbiente resolvido em {i + 1} episódios!")
        resolvido = True
        break

tot_time = time.time() - start
if not resolvido:      # o DQN pode piorar depois de um bom momento: volta para os melhores pesos vistos
    q_network.set_weights(melhores_pesos)
    print(f"pesos restaurados do episódio {melhor_episodio} (média dos últimos {JANELA_CHECKPOINT}: {melhor_media:.1f})")
print(f"\nTempo total: {tot_time:.1f} s ({tot_time / 60:.1f} min) | episódios: {len(total_point_history)} | resolvido: {resolvido}")
q_network.save("dqn_q_network.keras")

In [ ]:
def plot_history(pontos, janela=JANELA_MEDIA):
    """Pontos por episódio e média móvel (o plot_history do original)."""
    s = pd.Series(pontos)
    plt.figure(figsize=(8, 4))
    plt.plot(s, alpha=0.4, label="pontos do episódio")
    plt.plot(s.rolling(janela, min_periods=1).mean(), lw=2, label=f"média móvel ({janela})")
    plt.axhline(meta, ls="--", lw=1, label="meta")
    plt.xlabel("episódio"); plt.ylabel("pontos"); plt.legend(); plt.title(f"Treino em {ENV_ID}")
    plt.show()


plot_history(total_point_history)

## 7. Avaliação gulosa
O retorno de treino mistura exploração. O cartão pede o retorno médio **sem exploração** (ε = 0), em episódios com sementes que o treino não usou; para comparar configurações, repita o treino com **5 ou mais sementes**.

In [ ]:
retornos = []
for ep in range(EPISODIOS_AVALIACAO):
    s, _ = env.reset(seed=100_000 + ep)
    fim, ret = False, 0.0
    while not fim:
        a = get_action(q_valores(np.expand_dims(s, 0).astype("float32")), epsilon=0.0)
        s, r, terminated, truncated, _ = env.step(a)
        ret, fim = ret + r, terminated or truncated
    retornos.append(ret)
print(f"retorno guloso em {EPISODIOS_AVALIACAO} episódios: {np.mean(retornos):.1f} ± {np.std(retornos):.1f} (meta {meta})")

## 8. Ver o agente (opcional)

In [ ]:
if GRAVAR_GIF:
    from PIL import Image
    env_v = gym.make(ENV_ID, render_mode="rgb_array")      # renderizar exige pygame
    s, _ = env_v.reset(seed=123)
    quadros, fim = [Image.fromarray(env_v.render())], False
    while not fim and len(quadros) < 1000:
        a = get_action(q_valores(np.expand_dims(s, 0).astype("float32")))
        s, _, terminated, truncated, _ = env_v.step(a)
        fim = terminated or truncated
        quadros.append(Image.fromarray(env_v.render()))
    quadros[0].save("agente.gif", save_all=True, append_images=quadros[1::2], duration=40, loop=0)
    env_v.close()
    print("GIF salvo em agente.gif")
else:
    print("GRAVAR_GIF = False: seção pulada.")
env.close()

## 9. Próximos passos (segundo o cartão do sistema)
- O DQN é **sensível a hiperparâmetros e sementes**: taxa de aprendizado, tamanho do buffer e do lote, τ, γ e o decaimento de ε. Reporte média e desvio entre sementes.
- O cartão decai ε **por passo** e usa perda de Huber; o original decai por episódio e usa MSE. Para voltar ao esquema do original (por exemplo, no LunarLander), use os valores listados no comentário da configuração.
- **Prefira PPO ou SAC** para ações contínuas ou quando a estabilidade importar mais que a eficiência em amostras; **Q-learning tabular** se o espaço de estados for pequeno.

**Referências:** Mnih et al., *Human-level control through deep reinforcement learning*, Nature 518 (2015); Lillicrap et al., *Continuous Control with Deep Reinforcement Learning*, ICLR (2016); Mnih et al., *Playing Atari with Deep Reinforcement Learning*, arXiv:1312.5602 (2013).